In [10]:
import os
import joblib
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
import xgboost as xgb
from lightgbm import LGBMRegressor
import lightgbm as lgb
from sklearn.base import ClassifierMixin, RegressorMixin
from sklearn.preprocessing import FunctionTransformer

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

def load_pipeline_model(model_path: str) -> Pipeline:
    """
    Charge une Pipeline sklearn depuis un fichier .joblib dont le nom
    encode le label modélisé. Vérifie que l'objet chargé est bien
    une sklearn.pipeline.Pipeline et signale la présence d'un pas XGBRegressor.
    
    Parameters
    ----------
    model_path : str
        Chemin vers le fichier .joblib (ex. '.../xgbregressor_normalized_ENB2012_data.csv_l_heating_load.joblib')
    
    Returns
    -------
    pipeline : sklearn.pipeline.Pipeline
        La pipeline chargée.
    
    Raises
    ------
    ValueError
        Si le nom de fichier n'est pas au format attendu.
    TypeError
        Si l'objet chargé n'est pas une Pipeline.
    """
    # 1. Extraction du label depuis le nom de fichier
    basename    = os.path.basename(model_path)
    if basename.endswith('.joblib'):
        
        
        name_no_ext = basename[:-7]  # enlève '.joblib'
        try:
            # on ignore la première partie (classe) avant le premier '_'
            _, rest  = name_no_ext.split('_', 1)
            _, label = rest.split('.csv_', 1)
        except ValueError:
            raise ValueError(
                f"Le nom '{basename}' n'est pas au format attendu "
                "(<classe>_<nom.csv>_<label>.joblib)"
            )
        
        # 2. Chargement
        pipeline = joblib.load(model_path)
        
        # 3. Vérification du type
        if not isinstance(pipeline, Pipeline):
            raise TypeError(
                f"Objet chargé de type {type(pipeline).__name__} inattendu, "
                "attendu sklearn.pipeline.Pipeline"
            )
        
        # 4. inspection des estimateurs
        found = []
        for name, step in pipeline.named_steps.items():
            if isinstance(step, (ClassifierMixin, RegressorMixin)):
                found.append((name, step.__class__.__name__))
    
        if found:
            print("✅ Estimateurs détectés dans la pipeline :")
            for step_name, cls_name in found:
                print(f"  - {step_name}: {cls_name}")
        else:
            print("⚠️ Aucune étape de modélisation scikit-learn trouvée.")
    else:
        raise ValueError(f"Le fichier '{basename}' doit se terminer par '.joblib'")
    # 5. On retourne simplement la Pipeline
    print(f"Pipeline chargée pour le label '{label}'.")
    return pipeline

In [11]:
# vos autres transform et inverse-transform
TRANSFORM_FNS = {
    'none':   lambda x: x,
    'log':    np.log,
    'log1p':  np.log1p,
    'sqrt':   np.sqrt,
    'cbrt':   np.cbrt,
    'square': lambda x: x**2,
    'cube':   lambda x: x**3,
    'power4': lambda x: x**4,
}

INV_TRANSFORM_FNS = {
    'none':   lambda x: x,
    'log':    np.exp,
    'log1p':  np.expm1,
    'sqrt':   lambda x: x**2,
    'cbrt':   lambda x: x**3,
    'square': np.sqrt,
    'cube':   np.cbrt,
    'power4': lambda x: x**(1/4),
}

def apply_transform(series: pd.Series, method: str):
    """
    Retourne soit:
      - (serie_t, lambda) pour boxcox
      - serie_t pour les autres méthodes
    """
    if method == 'boxcox':
        # on s'assure que la série est strictement positive
        if (series <= 0).any():
            shift = -series.min() + 1e-6
            series = series + shift
            print(f"⚠ {method}: data shifted by +{shift:.6f} pour être >0")
        transformed, lmbda = boxcox(series.values)
        return transformed, lmbda
    else:
        fn = TRANSFORM_FNS.get(method)
        if fn is None:
            raise ValueError(f"Transformation inconnue: {method}")
        return fn(series.values)

def inverse_transform(y, method: str, lmbda=None):
    """
    Applique l'inverse de la transformation.
    Pour boxcox, il faut fournir λ.
    """
    if method == 'boxcox':
        if lmbda is None:
            raise ValueError("λ manquant pour l'inverse de boxcox")
        # inverse analytique
        y = np.asarray(y)
        if lmbda == 0:
            return np.exp(y)
        else:
            return (y * lmbda + 1) ** (1.0 / lmbda)
    else:
        fn = INV_TRANSFORM_FNS.get(method)
        if fn is None:
            raise ValueError(f"Transformation inverse inconnue: {method}")
        return fn(y)

l_transformations = ['log', 'sqrt', 'square', 'cube', 'power4', 'cbrt']
#f_transformations = ['log', 'sqrt', 'square']
#f_transformations = ['boxcox', 'log', 'sqrt', 'square', 'cube', 'power4', 'cbrt']#KO
#f_transformations = ['boxcox', 'log', 'sqrt', 'square']#KO
f_transformations = ['log', 'sqrt', 'square']#OK
#f_transformations = ['log', 'sqrt', 'square', 'cube', 'power4', 'cbrt']#KO
#f_transformations = ['log', 'sqrt', 'square', 'cube', 'cbrt']#KO
#f_transformations = ['log', 'sqrt', 'square', 'cube']#KO
#f_transformations = ['log', 'sqrt', 'square', 'cbrt']#KO
#f_transformations = ['log', 'sqrt', 'square', 'power4']#KO
#f_transformations = ['log', 'sqrt', 'power4']#KO
#f_transformations = ['log', 'sqrt', 'cube']#KO
#f_transformations = ['sqrt','square', 'cube']#KO
#f_transformations = ['log','square', 'cube']#KO
#l_transformations = ['log', 'sqrt', 'square']

label_transformer = FunctionTransformer(
    func=apply_transform,
    inverse_func=inverse_transform,
    validate=False
)
def get_polynomial_features(df: pd.DataFrame, target_col: str) -> list:
    """
    Sélectionne les features dont la corrélation absolue avec target_col est
    dans (0.33, 0.66), hors la colonne cible elle-même.
    """
    corrs = df.corr()[target_col].abs()
    feats = corrs[(corrs > 0.33) & (corrs < 0.66)].index.tolist()
    return [f for f in feats if f != target_col]

def create_polynomial_features(
    df: pd.DataFrame,
    features: list,
    transformations: list = f_transformations
) -> pd.DataFrame:
    """
    Pour chaque feature, génère des colonnes feat_method selon `transformations`.
    Pour 'boxcox', calcule d'abord le λ via boxcox_normmax, 
    stocke λ dans boxcox_lambdas[feat] et transforme.
    """
    df_poly = df.copy()
    for feat in features:
        x = df[feat].dropna().values
        for method in transformations:
            if method == 'boxcox':
                # on passe directement la Series à apply_transform
                series_t, lmbda = apply_transform(df[feat], method)
                boxcox_lambdas[feat] = lmbda
                df_poly[f"{feat}_boxcox"] = series_t
            else:
                # autres transformations via apply_transform
                df_poly[f"{feat}_{method}"] = apply_transform(df[feat], method)
    return df_poly

In [12]:
# Création d'un DataFrame d'exemple
df_example = pd.DataFrame({
    'f_relative_compactness': [0.76, 0.85, 0.92, 0.65, 0.88,0.82],
    'f_wall_area': [270.0, 350.5, 210.2, 299.9, 330.0,318.5],
    'f_overall_height': [3.5, 7.0, 3.0, 5.5, 6.0,7.0],
    'f_orientation': [2, 3, 4, 5, 2, 5],
    'f_glazing_area': [0.0, 0.1, 0.2, 0.4, 0.3,0.0]
})
df_example

,f_relative_compactness,f_wall_area,f_overall_height,f_orientation,f_glazing_area
0,0.76,270.0,3.5,2,0.0
1,0.85,350.5,7.0,3,0.1
2,0.92,210.2,3.0,4,0.2
3,0.65,299.9,5.5,5,0.4
4,0.88,330.0,6.0,2,0.3
5,0.82,318.5,7.0,5,0.0


In [13]:
features_name = ['f_relative_compactness', 'f_wall_area', 'f_overall_height', 'f_glazing_area', 'f_relative_compactness_log', 'f_relative_compactness_sqrt', 'f_relative_compactness_square', 'f_wall_area_log', 'f_wall_area_sqrt', 'f_wall_area_square', 'f_orientation_2', 'f_orientation_3', 'f_orientation_4', 'f_orientation_5']

def prepare_data(df: pd.DataFrame):
    """
    - Garde uniquement les colonnes dont le nom est contenu dans un feature de m_cooling_load
    - Renomme ces colonnes en les préfixant 'f_'
    - Convertit toutes les colonnes en float sauf 'f_orientation' en category
    - Extrait y_actual_heating_load et y_actual_cooling_load

    Returns:
        X: DataFrame prêt pour le modèle
        y_actual_heating_load: Series
        y_actual_cooling_load: Series
    """
    # 1) Récupère les noms de features LightGBM
    feature_names = features_name

    # 2) Filtre et préfixe
    orig_cols = [col for col in df.columns if any(col in feat for feat in feature_names)]
    X = df[orig_cols].copy()
    X.columns = ['f_' + col for col in orig_cols]

    # 3) Conversion des types
    for col in X.columns:
        if col == 'f_orientation':
            # on garde l'orientation en category
            X[col] = X[col].astype('category')
        else:
            # conversion numérique stricte
            X[col] = pd.to_numeric(X[col], errors='raise')

    # 4) Extraction des y
    y_actual_heating_load = pd.to_numeric(df['heating_load'], errors='raise')
    y_actual_cooling_load = pd.to_numeric(df['cooling_load'], errors='raise')

    return X, y_actual_heating_load, y_actual_cooling_load

In [16]:
# 1. Charger le fichier CSV avec séparateur ';'
df = pd.read_csv('../data/normalized_csv/normalized_ENB2012_data.csv', sep=';',decimal=',')

# 2) préparation X et y
X_prepared, y_h, y_c = prepare_data(df)


# 2) génère les poly-features comme en train
feats = get_polynomial_features(df, 'l_cooling_load')  # même fonction qu’en train
df_poly = create_polynomial_features(df, feats, transformations=f_transformations)




KeyError: 'l_cooling_load'

In [8]:
model_path_l_cooling_load = 'saved_models/lgbmregressor_normalized_ENB2012_data.csv_l_cooling_load.joblib'
model_path_l_heating_load = 'saved_models/xgbregressor_normalized_ENB2012_data.csv_l_heating_load.joblib'


pipeline_cooling_load = load_pipeline_model(model_path_l_cooling_load)
pipeline_heating_load = load_pipeline_model(model_path_l_heating_load)


# 4) Prédiction (le Pipeline prétraitera puis appellera le booster)
preds_log_cool = pipeline_cooling_load.predict(X)
preds_log_heat = pipeline_heating_load.predict(X)

# 5) Inverse du log (ou log1p/boxcox selon le label)
preds_cool = np.exp(preds_log_cool)   # ou np.expm1 si log1p
preds_heat = np.exp(preds_log_heat)

# 6) RMSE
rmse_cool = np.sqrt(mean_squared_error(model_path_l_cooling_load, preds_cool))
rmse_heat = np.sqrt(mean_squared_error(y_actual_heating_load, preds_heat))

print(f"RMSE Cooling : {rmse_cool:.4f}")
print(f"RMSE Heating : {rmse_heat:.4f}")


✅ Estimateurs détectés dans la pipeline :
  - model: LGBMRegressor
Pipeline chargée pour le label 'l_cooling_load'.
✅ Estimateurs détectés dans la pipeline :
  - model: XGBRegressor
Pipeline chargée pour le label 'l_heating_load'.


/opt/conda/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


InvalidParameterError: The 'y_true' parameter of mean_squared_error must be an array-like. Got 'saved_models/lgbmregressor_normalized_ENB2012_data.csv_l_cooling_load.joblib' instead.

In [1]:


# 1. Charger le Booster depuis le fichier .txt
m_cooling_load = lgb.Booster(model_file='saved_models/lgbmregressor_normalized_ENB2012_data.csv_l_cooling_load_booster.txt')
m_heating_load=XGBRegressor()
m_heating_load = xgb.Booster(model_file='saved_models/xgbregressor_normalized_ENB2012_data.csv_l_heating_load_booster.txt')

# 2) Récupérer la liste des noms de features :
f_cooling_load = m_cooling_load.feature_name()

# 3) L’afficher :
print(f_cooling_load)




NameError: name 'lgb' is not defined

In [31]:
# 1) Cooling load (LightGBM accepte le DataFrame directement)
df_with_predictions = df_transformed.copy()
# liste des colonnes initiales (sans les colonnes de prédiction)
feature_cols = m_cooling_load.feature_name()  # ou la liste que vous aviez utilisée à l'entraînement

# reconstruire votre DMatrix sur ces colonnes seules
dmat_heating = xgb.DMatrix(
    df_with_predictions[feature_cols],
    feature_names=feature_cols
)

# 2) Calculez les prédictions en log
preds_cooling_log = m_cooling_load.predict(df_with_predictions[feature_cols])
preds_heating_log = m_heating_load.predict(dmat_heating)

# 3) Appliquez l’inverse du log
df_with_predictions['predicted_cooling_load'] = np.exp(preds_cooling_log)
df_with_predictions['predicted_heating_load'] = np.exp(preds_heating_log)

# 3) Vérification
display(df_with_predictions)

,f_relative_compactness,f_wall_area,f_overall_height,f_glazing_area,f_relative_compactness_log,f_relative_compactness_sqrt,f_relative_compactness_square,f_wall_area_log,f_wall_area_sqrt,f_wall_area_square,f_orientation_2,f_orientation_3,f_orientation_4,f_orientation_5,predicted_cooling_load,predicted_heating_load
0,0.76,270.0,3.5,0.0,-0.274437,0.871780,0.5776,5.598422,16.431677,72900.00,1,0,0,0,30.470252,22.236917
1,0.85,350.5,7.0,0.1,-0.162519,0.921954,0.7225,5.859361,18.721645,122850.25,0,1,0,0,32.696484,22.227182
2,0.92,210.2,3.0,0.2,-0.083382,0.959166,0.8464,5.348059,14.498276,44184.04,0,0,1,0,33.034074,25.053015
3,0.65,299.9,5.5,0.4,-0.430783,0.806226,0.4225,5.703449,17.317621,89940.01,0,0,0,1,30.822578,24.620998
4,0.88,330.0,6.0,0.3,-0.127833,0.938083,0.7744,5.799093,18.165902,108900.00,1,0,0,0,33.110720,24.625748
5,0.82,318.5,7.0,0.0,-0.198451,0.905539,0.6724,5.763622,17.846568,101442.25,0,0,0,1,30.626511,22.178404


In [75]:
from sklearn.metrics import mean_squared_error

# 1) Prédictions en log
preds_log_cool = m_cooling_load.predict(X)
dmat = xgb.DMatrix(X, feature_names=X.columns.tolist())
preds_log_heat = m_heating_load.predict(dmat)

# 2) Back-transform
preds_cool = np.exp(preds_log_cool)
preds_heat = np.exp(preds_log_heat)

# 3) Calcul du MSE, puis RMSE
mse_cool = mean_squared_error(y_actual_cooling_load, preds_cool)
rmse_cool = np.sqrt(mse_cool)

mse_heat = mean_squared_error(y_actual_heating_load, preds_heat)
rmse_heat = np.sqrt(mse_heat)

print(f"RMSE Cooling Load: {rmse_cool:.4f}")
print(f"RMSE Heating Load: {rmse_heat:.4f}")

RMSE Cooling Load: 10.9631
RMSE Heating Load: 9.9305


## 